# IMDb Sentiment Classification using Deep Learning

This notebook builds an end-to-end binary sentiment classifier using TensorFlow/Keras. The model includes text vectorization, an embedding layer, and global average pooling.

## Problem statement

Build a deep-learning application that reads an IMDb movie review and predicts whether its sentiment is positive or negative.

## Download the dataset

Download the [IMDb Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews), extract it, and place `IMDB Dataset.csv` inside a `data` folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
tf.keras.utils.set_random_seed(SEED)

## 1. Load and inspect the data

In [ ]:
DATA_PATH = Path("data/IMDB Dataset.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass distribution:\n", df["sentiment"].value_counts())

In [ ]:
sns.countplot(data=df, x="sentiment")
plt.title("IMDb sentiment distribution")
plt.show()

## 2. Clean and prepare the labels

In [ ]:
df = df.dropna(subset=["review", "sentiment"]).drop_duplicates().copy()
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})
df[["review", "sentiment", "label"]].head()

## 3. Create stratified train, validation, and test sets

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df["review"].astype(str),
    df["label"].astype("float32"),
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print(len(X_train), len(X_val), len(X_test))

## 4. Build TensorFlow datasets

In [ ]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(texts, labels, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((texts.to_numpy(), labels.to_numpy()))
    if shuffle:
        dataset = dataset.shuffle(len(texts), seed=SEED)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train, y_train, shuffle=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

## 5. Adapt the TextVectorization layer using training text only

In [ ]:
MAX_TOKENS = 20_000
SEQUENCE_LENGTH = 300

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
vectorizer.adapt(X_train.to_numpy())
print("Vocabulary size:", len(vectorizer.get_vocabulary()))

## 6. Build and compile the neural network

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string, name="review"),
    vectorizer,
    tf.keras.layers.Embedding(MAX_TOKENS, 64),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid", name="positive_probability"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
)
model.summary()

## 7. Train with early stopping

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True
    )
]

history = model.fit(
    train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks
)

## 8. Plot training history

In [ ]:
history_df = pd.DataFrame(history.history)
history_df[["loss", "val_loss"]].plot(title="Training and validation loss")
plt.show()
history_df[["accuracy", "val_accuracy"]].plot(title="Training and validation accuracy")
plt.show()

## 9. Evaluate on unseen test data

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True)
test_results

In [ ]:
probabilities = model.predict(X_test.to_numpy(), batch_size=BATCH_SIZE).reshape(-1)
predictions = (probabilities >= 0.5).astype(int)
print(classification_report(y_test.astype(int), predictions, target_names=["Negative", "Positive"]))

matrix = confusion_matrix(y_test.astype(int), predictions)
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 10. Test custom reviews

In [ ]:
sample_reviews = np.array([
    "The acting was excellent and the story kept me engaged.",
    "This movie was boring, predictable, and far too long.",
])
sample_probabilities = model.predict(sample_reviews, verbose=0).reshape(-1)
for review, probability in zip(sample_reviews, sample_probabilities):
    label = "Positive" if probability >= 0.5 else "Negative"
    confidence = probability if probability >= 0.5 else 1 - probability
    print(f"{label} ({confidence:.1%}): {review}")

## 11. Save and reload the complete model

In [ ]:
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODEL_PATH = ARTIFACTS_DIR / "sentiment_dl.keras"
model.save(MODEL_PATH)
print("Saved model to:", MODEL_PATH)

In [ ]:
reloaded_model = tf.keras.models.load_model(MODEL_PATH)
reloaded_model.predict(np.array(["A wonderful and memorable film."]), verbose=0)